In [3]:
import requests
import csv
import os
import subprocess
import time
import pandas as pd


In [4]:
def run_mafft_alignment(input_fasta, output_alignment):
    """
    Run MAFFT alignment on the given input FASTA file and save the output to the specified file.

    Parameters:
    - input_fasta (str): Path to the input FASTA file.
    - output_alignment (str): Path to the output alignment file.
    """


    # Create the output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_alignment), exist_ok=True)

    # Run the MAFFT alignment command
    try:
        subprocess.run(
            ["mafft", input_fasta],
            stdout=open(output_alignment, "w"),
            stderr=subprocess.PIPE,
            check=True
        )
        print(f"Alignment completed successfully. Output written to {output_alignment}")
    except subprocess.CalledProcessError as e:
        print(f"Error during alignment: {e.stderr.decode()}")

In [6]:
# List of human gene ENSTs
lambert_TFs_ensg = pd.read_csv("../data/lambert_TF_ensg.csv", header = None)
lambert_TFs_ensg

,0
0,ENSG00000137203
1,ENSG00000008196
2,ENSG00000087510
3,ENSG00000008197
4,ENSG00000116819
...,...
1634,ENSG00000177683
1635,ENSG00000174796
1636,ENSG00000184436
1637,ENSG00000161277


In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Convert the DataFrame column to a list of gene IDs
gene_list = lambert_TFs_ensg[0].tolist()

output_dir = "../output/ensembl_orthologs/ortholog_fastas"
os.makedirs(output_dir, exist_ok=True)

params = {
    "type": "orthologues",
    "sequence": "protein",
    "aligned": 1,
}

headers = {"Accept": "application/json"}

def process_gene(gene_id):
    try:
        url = f"https://rest.ensembl.org/homology/id/human/{gene_id}"
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
    except requests.HTTPError as e:
        return f"Failed to fetch {gene_id}: {e}"

    gene_data = data.get("data", [])
    if not gene_data:
        return f"No homology data for {gene_id}"

    homologies = gene_data[0].get("homologies", [])
    fasta_file = os.path.join(output_dir, f"{gene_id}_orthologues.fasta")
    
    with open(fasta_file, "w") as fasta_out:
        # Write reference sequence first
        ref_written = False
        for homology in homologies:
            source = homology.get("source")
            if source and source.get("id") == gene_id and "align_seq" in source:
                seq_id = f"{source['id']}|{source.get('protein_id','')}|{source['species']}"
                seq = source['align_seq'].replace("-", "")  # remove dashes
                fasta_out.write(f">{seq_id}\n{seq}\n\n")
                ref_written = True
                break

        # If still not found, fetch unaligned protein sequence
        if not ref_written:
            seq_url = f"https://rest.ensembl.org/sequence/id/{gene_id}?type=protein"
            seq_resp = requests.get(seq_url, headers={"Accept": "text/plain"})
            if seq_resp.ok:
                fasta_out.write(f">{gene_id}|unaligned|homo_sapiens\n{seq_resp.text}\n")
            else:
                return f"Could not fetch reference sequence for {gene_id}"

        # Write orthologues
        for homology in homologies:
            if "ortholog" in homology["type"]:
                target = homology["target"]
                if target["species"] != "homo_sapiens" and "align_seq" in target:
                    seq_id = f"{target['id']}|{target.get('protein_id','')}|{target['species']}"
                    seq = target['align_seq'].replace("-", "")  # remove dashes
                    fasta_out.write(f">{seq_id}\n{seq}\n\n")

    run_mafft_alignment(input_fasta=fasta_file,
                        output_alignment=f"../output/ensembl_orthologs/mafft_alignments/{gene_id}_alignment.fasta")
    return f"FASTA written for {gene_id}: {fasta_file}"

# Use ThreadPoolExecutor for parallel processing
max_workers = 4  # Adjust the number of workers as needed
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_gene, gene_id): gene_id for gene_id in gene_list}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing genes"):
        result = future.result()
        print(result)

Processing genes:   0%|          | 0/1639 [00:04<?, ?it/s]


Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000008196_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000008197_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000137203_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000087510_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000116819_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000116017_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/ENSG00000205143_alignment.fasta
Alignment completed successfully. Output written to ../output/ensembl_orthologs/mafft_alignments/

KeyboardInterrupt: 